# WaterMelonBraker: YOLOv8s fine-tune for watermelon detection (Google Colab版)

Roboflow の vietnam-fruit-in-lab/watermelon-n4rrw データセットを YOLOv8s で fine-tune し、Quest 3 の Sentis 用に ONNX エクスポートするノートブック。

**ランタイム**: Colab T4 GPU（無料枠）で 30〜50分

> 💡 Colab の無料 GPU 枠切れ時は `train_watermelon_kaggle.ipynb` (Kaggle版) を使用。

## 事前準備

1. Colab 上部メニュー → `ランタイム` → `ランタイムのタイプを変更` → `T4 GPU` を選択
2. 左サイドバーの 🔑 アイコン（シークレット）を開く
3. `+ 新しいシークレットを追加` で `ROBOFLOW_API_KEY` という名前で API Key を保存
4. トグルを ON にして「ノートブックからのアクセス」を許可


## Step 1: 環境確認

In [ ]:
!nvidia-smi

## Step 2: 必要なパッケージをインストール

In [ ]:
!pip install -q ultralytics roboflow onnx onnxsim

## Step 3: データセットのダウンロード

vietnam-fruit-in-lab の watermelon-n4rrw を使用（106枚、畑・遠距離画像あり）。

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace('vietnam-fruit-in-lab').project('watermelon-n4rrw')
version = project.version(1)
dataset = version.download('yolov8')

print('Dataset location:', dataset.location)
!ls {dataset.location}

## Step 4: data.yaml を確認

In [ ]:
!cat {dataset.location}/data.yaml

## Step 5: YOLOv8s を fine-tune

**設定**

- `model='yolov8s.pt'`: small (11M params)、遠距離検出に適する
- `epochs=100`: 106枚と少ないデータなので多めに
- `imgsz=640`: 標準
- `batch=16`: T4 GPU (16GB)で余裕
- `patience=20`: 20エポック改善なければ早期終了
- `augment=True`: augmentation強化(少ないデータで過学習を避ける)

T4 GPU で 30〜50分。

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project='runs/train',
    name='watermelon',
    exist_ok=True,
    augment=True,
)

## Step 6: 学習結果の可視化

In [ ]:
from IPython.display import Image, display

display(Image('runs/train/watermelon/results.png'))
display(Image('runs/train/watermelon/confusion_matrix.png'))

## Step 7: 検証セットで動作確認

In [ ]:
best_model = YOLO('runs/train/watermelon/weights/best.pt')
metrics = best_model.val(data=f'{dataset.location}/data.yaml')
print(f'mAP@50:    {metrics.box.map50:.4f}')
print(f'mAP@50-95: {metrics.box.map:.4f}')

## Step 8: ONNX 形式にエクスポート

Meta 公式サンプルの `SentisModelEditorConverter.cs` が期待する出力形状 (1, 84, N) の標準 YOLOv8 ONNX。

- `imgsz=640`: Sentis 入力サイズと一致
- `opset=15`: Sentis / Unity Inference Engine が安定サポートする opset
- `simplify=True`: グラフ最適化
- `dynamic=False`: 静的shape
- `nms=False`: NMS は Sentis 側の SentisModelEditorConverter.cs で組み込む

In [ ]:
onnx_path = best_model.export(
    format='onnx',
    imgsz=640,
    opset=15,
    simplify=True,
    dynamic=False,
    nms=False,
)
print('ONNX saved to:', onnx_path)

## Step 9: クラス名リストを出力

In [ ]:
class_names = best_model.names
print('Class count:', len(class_names))

with open('SentisYoloClasses.txt', 'w') as f:
    for i in range(len(class_names)):
        f.write(class_names[i] + '\n')

print('\n=== SentisYoloClasses.txt ===')
with open('SentisYoloClasses.txt') as f:
    print(f.read())

watermelon_id = [i for i, name in class_names.items() if 'watermelon' in name.lower()]
print(f'watermelon class ID: {watermelon_id}')

## Step 10: 成果物をダウンロード

- `watermelon.onnx`: Unity Sentis に投入する ONNX モデル
- `SentisYoloClasses.txt`: クラス名リスト
- `watermelon_best.pt`: 元の PyTorch weight (念のため保存)

In [ ]:
import shutil
from google.colab import files

shutil.copy(onnx_path, 'watermelon.onnx')
shutil.copy('runs/train/watermelon/weights/best.pt', 'watermelon_best.pt')

files.download('watermelon.onnx')
files.download('SentisYoloClasses.txt')
files.download('watermelon_best.pt')